# 第2章：颜色空间的转换

## 编程实践：颜色空间转换与颜色传递

---

## 一、颜色空间基础

### 1.1 什么是颜色空间？

颜色空间是用**数学坐标**来表示颜色的模型。每种颜色空间都有其特定的物理含义和适用场景。

### 1.2 三种常用颜色空间

#### RGB 空间
- **物理含义**：基于红(R)、绿(G)、蓝(B)三原色光的加法混合
- **特点**：线性空间，符合物理显示原理
- **范围**：R,G,B ∈ [0, 255]（8位）
- **适用**：显示器、相机等显示设备

#### HSV 空间
- **物理含义**：
  - **H (Hue)**：色调，0°~360°，表示颜色种类（红=0°, 绿=120°, 蓝=240°）
  - **S (Saturation)**：饱和度，0~1，表示颜色纯度
  - **V (Value)**：明度，0~1，表示亮度
- **特点**：更符合人眼感知
- **适用**：颜色分割、颜色检测

#### Lab 空间
- **物理含义**：CIE L*a*b* 颜色空间，模拟人眼感知
  - **L**：亮度，0~100
  - **a**：红绿通道 (-128~127)
  - **b**：黄蓝通道 (-128~127)
- **特点**：感知均匀（欧氏距离≈感知差异）
- **适用**：颜色比较、颜色传递

### 1.3 颜色空间转换关系

```
  RGB ──→ HSV    (非线性变换)
  RGB ──→ Lab    (非线性变换，需经过 XYZ)
  HSV ──→ RGB    (逆变换)
  Lab ──→ RGB    (逆变换，需经过 XYZ)
```


## 二、实现要求

### 实践1：颜色空间转换
> 将彩色图像从 RGB 转到 HSV/Lab 空间，将每个通道单独保存为图像，再转换回 RGB。除 OpenCV 读写函数外，其余代码手写。

### 实践2：颜色传递
> 实现两张彩色图像之间的颜色传递算法。源图的颜色风格传递给目标图。除 OpenCV 读写函数外，其余代码手写。


In [ ]:
# 导入库 (仅 cv2 用于图像读写)
import cv2
import numpy as np
import math

print(f"OpenCV 版本: {cv2.__version__}")

In [ ]:
# 生成本章所需的测试图像
import numpy as np

print("正在生成测试图像...")

# 1. 彩色测试图像
h, w = 400, 600
img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    for x in range(w):
        img[y, x, 0] = int(180 * x / w)
        img[y, x, 1] = int(200 * y / h)
        img[y, x, 2] = int(120 + 80 * (x + y) / (w + h))
cv2.rectangle(img, (50, 50), (150, 150), (0, 0, 255), -1)
cv2.rectangle(img, (200, 50), (300, 150), (0, 255, 0), -1)
cv2.rectangle(img, (350, 50), (450, 150), (255, 0, 0), -1)
cv2.circle(img, (150, 250), 60, (255, 0, 255), -1)
cv2.circle(img, (350, 280), 80, (128, 255, 128), -1)
cv2.imwrite("color_image.jpg", img)

# 2. 暖色调源图像
h2, w2 = 300, 400
src = np.zeros((h2, w2, 3), dtype=np.uint8)
for y in range(h2):
    for x in range(w2):
        src[y, x, 0] = int(30 + 50 * y / h2)
        src[y, x, 1] = int(100 + 80 * x / w2)
        src[y, x, 2] = int(180 + 60 * (x + y) / (w2 + h2))
cv2.circle(src, (200, 150), 80, (0, 100, 200), -1)
cv2.imwrite("transfer_source.jpg", src)

# 3. 冷色调目标图像
dst = np.zeros((h2, w2, 3), dtype=np.uint8)
for y in range(h2):
    for x in range(w2):
        dst[y, x, 0] = int(180 + 60 * x / w2)
        dst[y, x, 1] = int(100 + 50 * y / h2)
        dst[y, x, 2] = int(40 + 40 * (x + y) / (w2 + h2))
cv2.rectangle(dst, (50, 50), (200, 250), (200, 100, 50), -1)
cv2.imwrite("transfer_target.jpg", dst)

print("所有测试图像已生成!")

In [ ]:
def rgb_to_hsv_manual(r, g, b):
    """
    手写 RGB → HSV 转换 (单个像素)
    RGB 范围: 0~255
    HSV 范围: H: 0~360, S: 0~1, V: 0~1
    """
    # 归一化到 [0, 1]
    r_norm = r / 255.0
    g_norm = g / 255.0
    b_norm = b / 255.0
    
    # 找最大值和最小值
    cmax = max(r_norm, g_norm, b_norm)  # 最大分量
    cmin = min(r_norm, g_norm, b_norm)  # 最小分量
    delta = cmax - cmin
    
    # ===== 计算 H (色调) =====
    if delta == 0:
        h = 0  # 灰度情况
    elif cmax == r_norm:
        # 红色在最大值位置
        h = 60 * (((g_norm - b_norm) / delta) % 6)
    elif cmax == g_norm:
        # 绿色在最大值位置
        h = 60 * (((b_norm - r_norm) / delta) + 2)
    else:  # cmax == b_norm
        # 蓝色在最大值位置
        h = 60 * (((r_norm - g_norm) / delta) + 4)
    
    # ===== 计算 S (饱和度) =====
    if cmax == 0:
        s = 0
    else:
        s = delta / cmax
    
    # ===== 计算 V (明度) =====
    v = cmax
    
    return h, s, v


def hsv_to_rgb_manual(h, s, v):
    """
    手写 HSV → RGB 转换 (单个像素)
    HSV 范围: H: 0~360, S: 0~1, V: 0~1
    RGB 范围: 0~255
    """
    # 处理 H 为 0 的情况
    h = h % 360
    
    c = v * s          # 色度
    x = c * (1 - abs((h / 60.0) % 2 - 1))  # 中间分量
    m = v - c          # 匹配值
    
    # 根据 H 的角度确定 RGB 顺序
    if h < 60:
        r, g, b = c, x, 0
    elif h < 120:
        r, g, b = x, c, 0
    elif h < 180:
        r, g, b = 0, c, x
    elif h < 240:
        r, g, b = 0, x, c
    elif h < 300:
        r, g, b = x, 0, c
    else:
        r, g, b = c, 0, x
    
    # 加上匹配值并转换到 [0, 255]
    R = int((r + m) * 255)
    G = int((g + m) * 255)
    B = int((b + m) * 255)
    
    return R, G, B

In [ ]:
def rgb_to_lab_manual(r, g, b):
    """
    手写 RGB → Lab 转换 (单个像素)
    需要先转换到 XYZ 空间
    """
    # Step 1: 归一化到 [0, 1]
    r_norm = r / 255.0
    g_norm = g / 255.0
    b_norm = b / 255.0
    
    # Step 2: sRGB → 线性 RGB (gamma 反变换)
    def srgb_to_linear(c):
        if c <= 0.04045:
            return c / 12.92
        else:
            return ((c + 0.055) / 1.055) ** 2.4
    
    R = srgb_to_linear(r_norm)
    G = srgb_to_linear(g_norm)
    B = srgb_to_linear(b_norm)
    
    # Step 3: 线性 RGB → XYZ (D65 光源)
    # 转换矩阵 (sRGB → XYZ)
    X = 0.4124564 * R + 0.3575761 * G + 0.1804375 * B
    Y = 0.2126729 * R + 0.7151522 * G + 0.0721750 * B
    Z = 0.0193339 * R + 0.1191920 * G + 0.9503041 * B
    
    # Step 4: XYZ → Lab (D65 白点)
    # D65 白点值
    Xn, Yn, Zn = 0.95047, 1.00000, 1.08883
    
    # 归一化
    x = X / Xn
    y = Y / Yn
    z = Z / Zn
    
    # 非线性变换
    epsilon = 216.0 / 24389.0   # 0.008856
    kappa = 24389.0 / 27.0      # 903.3
    
    def f(t):
        if t > epsilon:
            return t ** (1.0 / 3.0)
        else:
            return (kappa * t + 16.0) / 116.0
    
    fx, fy, fz = f(x), f(y), f(z)
    
    # 计算 Lab
    L = 116.0 * fy - 16.0
    a = 500.0 * (fx - fy)
    b_val = 200.0 * (fy - fz)  # 注意: b 已被占用
    
    return L, a, b_val


def lab_to_rgb_manual(L, a, b_val):
    """
    手写 Lab → RGB 转换 (单个像素)
    """
    # Step 1: Lab → XYZ
    fx = (L + 16.0) / 116.0
    fy = fx - a / 500.0
    fz = fy + b_val / 200.0
    
    epsilon = 216.0 / 24389.0
    kappa = 24389.0 / 27.0
    
    def f_inv(t):
        t3 = t ** 3
        if t3 > epsilon:
            return t3
        else:
            return (116.0 * t - 16.0) / kappa
    
    # D65 白点
    Xn, Yn, Zn = 0.95047, 1.00000, 1.08883
    
    X = f_inv(fx) * Xn
    Y = f_inv(fy) * Yn
    Z = f_inv(fz) * Zn
    
    # Step 2: XYZ → 线性 RGB
    R =  3.2404542 * X - 1.5371385 * Y - 0.4985314 * Z
    G = -0.9692660 * X + 1.8760108 * Y + 0.0415560 * Z
    B =  0.0556434 * X - 0.2040259 * Y + 1.0572252 * Z
    
    # Step 3: 线性 RGB → sRGB
    def linear_to_srgb(c):
        if c <= 0.0031308:
            return 12.92 * c
        else:
            return 1.055 * (c ** (1.0 / 2.4)) - 0.055
    
    R = int(max(0, min(255, linear_to_srgb(R) * 255)))
    G = int(max(0, min(255, linear_to_srgb(G) * 255)))
    B = int(max(0, min(255, linear_to_srgb(B) * 255)))
    
    return R, G, B

In [ ]:
def convert_image_rgb_to_hsv(image):
    """
    对整张图像进行 RGB→HSV 转换 (逐像素)
    输入/输出都是 OpenCV 的 BGR 格式
    """
    h, w = image.shape[:2]
    result = image.copy()
    
    for y in range(h):
        for x in range(w):
            # OpenCV 是 BGR 顺序
            b, g, r = image[y, x, 0], image[y, x, 1], image[y, x, 2]
            h_val, s_val, v_val = rgb_to_hsv_manual(r, g, b)
            
            # 存储 HSV (为了保存为图像, 映射到 [0,255])
            result[y, x, 0] = int(h_val * 255 / 360)  # H: 0~360 → 0~255
            result[y, x, 1] = int(s_val * 255)          # S: 0~1 → 0~255
            result[y, x, 2] = int(v_val * 255)          # V: 0~1 → 0~255
    
    return result


def convert_image_rgb_to_lab(image):
    """
    对整张图像进行 RGB→Lab 转换 (逐像素)
    """
    h, w = image.shape[:2]
    result = image.copy().astype('float32')
    
    for y in range(h):
        for x in range(w):
            b, g, r = image[y, x, 0], image[y, x, 1], image[y, x, 2]
            L, a, b_val = rgb_to_lab_manual(r, g, b)
            
            # 存储 Lab (映射到 [0,255])
            result[y, x, 0] = np.clip(int(L * 255 / 100), 0, 255)  # L: 0~100
            result[y, x, 1] = np.clip(int((a + 128) * 255 / 256), 0, 255)  # a: -128~127
            result[y, x, 2] = np.clip(int((b_val + 128) * 255 / 256), 0, 255)  # b: -128~127
    
    return result.astype('uint8')



In [ ]:
# ==================== 实践1: 颜色空间转换 ====================

# 读取彩色图像
img = cv2.imread("color_image.jpg", cv2.IMREAD_COLOR)

if img is not None:
    print("读取图像成功!")
    h, w = img.shape[:2]
    print(f"尺寸: {w}x{h}")
    
    # --- RGB → HSV ---
    print("\n正在进行 RGB → HSV 转换...")
    hsv_img = convert_image_rgb_to_hsv(img)
    
    # 保存 HSV 各通道
    cv2.imwrite("h_channel.jpg", np.clip(hsv_img[:,:,0], 0, 255).astype(np.uint8))  # H 通道
    cv2.imwrite("s_channel.jpg", np.clip(hsv_img[:,:,1], 0, 255).astype(np.uint8))  # S 通道
    cv2.imwrite("v_channel.jpg", np.clip(hsv_img[:,:,2], 0, 255).astype(np.uint8))  # V 通道
    print("HSV 通道已保存: h_channel.jpg, s_channel.jpg, v_channel.jpg")
    
    # --- RGB → Lab ---
    print("\n正在进行 RGB → Lab 转换...")
    lab_img = convert_image_rgb_to_lab(img)
    
    # 保存 Lab 各通道
    cv2.imwrite("l_channel.jpg", lab_img[:,:,0])  # L 通道
    cv2.imwrite("a_channel.jpg", lab_img[:,:,1])  # a 通道
    cv2.imwrite("b_channel.jpg", lab_img[:,:,2])  # b 通道
    print("Lab 通道已保存: l_channel.jpg, a_channel.jpg, b_channel.jpg")
    
    # --- 从 HSV 读取通道并转回 RGB ---
    print("\n正在从 HSV 通道还原 RGB...")
    h_ch = cv2.imread("h_channel.jpg", cv2.IMREAD_GRAYSCALE)
    s_ch = cv2.imread("s_channel.jpg", cv2.IMREAD_GRAYSCALE)
    v_ch = cv2.imread("v_channel.jpg", cv2.IMREAD_GRAYSCALE)
    
    # 重组 HSV 并转回 RGB
    restored_from_hsv = np.zeros_like(img)
    for y in range(h):
        for x in range(w):
            h_val = float(h_ch[y, x]) * 360.0 / 255.0
            s_val = s_ch[y, x] / 255.0
            v_val = v_ch[y, x] / 255.0
            r, g, b = hsv_to_rgb_manual(h_val, s_val, v_val)
            restored_from_hsv[y, x] = [b, g, r]  # BGR 格式
    
    cv2.imwrite("restored_from_hsv.jpg", restored_from_hsv)
    print("已保存: restored_from_hsv.jpg")
    
    # --- 从 Lab 读取通道并转回 RGB ---
    print("\n正在从 Lab 通道还原 RGB...")
    l_ch = cv2.imread("l_channel.jpg", cv2.IMREAD_GRAYSCALE)
    a_ch = cv2.imread("a_channel.jpg", cv2.IMREAD_GRAYSCALE)
    b_ch = cv2.imread("b_channel.jpg", cv2.IMREAD_GRAYSCALE)
    
    restored_from_lab = np.zeros_like(img)
    for y in range(h):
        for x in range(w):
            L = float(l_ch[y, x]) * 100.0 / 255.0
            A = float(a_ch[y, x]) * 256.0 / 255.0 - 128.0
            B_val = float(b_ch[y, x]) * 256.0 / 255.0 - 128.0
            r, g, b = lab_to_rgb_manual(L, A, B_val)
            restored_from_lab[y, x] = [b, g, r]  # BGR 格式
    
    cv2.imwrite("restored_from_lab.jpg", restored_from_lab)
    print("已保存: restored_from_lab.jpg")
else:
    print("读取图像失败!")




In [ ]:
import matplotlib.pyplot as plt

if img is not None:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # 原图
    axes[0, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    # HSV 通道
    axes[0, 1].imshow(hsv_img[:,:,0], cmap='gray')
    axes[0, 1].set_title('H Channel')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(hsv_img[:,:,1], cmap='gray')
    axes[0, 2].set_title('S Channel')
    axes[0, 2].axis('off')
    
    axes[0, 3].imshow(hsv_img[:,:,2], cmap='gray')
    axes[0, 3].set_title('V Channel')
    axes[0, 3].axis('off')
    
    # Lab 通道
    axes[1, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title('Original')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(lab_img[:,:,0], cmap='gray')
    axes[1, 1].set_title('L Channel')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(lab_img[:,:,1], cmap='gray')
    axes[1, 2].set_title('a Channel')
    axes[1, 2].axis('off')
    
    axes[1, 3].imshow(lab_img[:,:,2], cmap='gray')
    axes[1, 3].set_title('b Channel')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

## 三、颜色传递算法

### 3.1 原理
颜色传递(Color Transfer)的目标是将**源图像的颜色风格**迁移到**目标图像**上。

经典算法 (Reinhard et al., 2001)：
1. 将源图和目标图转换到 **Lab 颜色空间**（感知均匀）
2. 分别计算两张图在 L、a、b 三个通道的**均值**和**标准差**
3. 对每个通道进行变换：
```
   output = (input - target_mean) * (source_std / target_std) + source_mean
```
4. 将结果从 Lab 转换回 RGB

### 3.2 实现步骤
```
读取源图和目标图 → RGB转Lab → 计算统计量 → 颜色变换 → Lab转RGB → 保存
```


In [ ]:
def color_transfer_manual(source_img, target_img):
    """
    手写颜色传递算法 (基于 Reinhard 方法)
    将 source_img 的颜色风格传递到 target_img
    """
    h, w = target_img.shape[:2]
    
    # Step 1: 将源图和目标图转到 Lab 空间
    print("  转换源图到 Lab 空间...")
    source_lab = convert_image_rgb_to_lab(source_img).astype('float64')
    
    print("  转换目标图到 Lab 空间...")
    target_lab = convert_image_rgb_to_lab(target_img).astype('float64')
    
    # Step 2: 计算源图各通道的均值和标准差
    source_means = []
    source_stds = []
    for c in range(3):
        source_means.append(source_lab[:,:,c].mean())
        source_stds.append(source_lab[:,:,c].std())
    
    print(f"  源图 Lab 均值: L={source_means[0]:.1f}, a={source_means[1]:.1f}, b={source_means[2]:.1f}")
    
    # Step 3: 计算目标图各通道的均值和标准差
    target_means = []
    target_stds = []
    for c in range(3):
        target_means.append(target_lab[:,:,c].mean())
        target_stds.append(target_lab[:,:,c].std())
    
    print(f"  目标图 Lab 均值: L={target_means[0]:.1f}, a={target_means[1]:.1f}, b={target_means[2]:.1f}")
    
    # Step 4: 对目标图每个通道进行颜色传递变换
    result_lab = target_lab.copy()
    for c in range(3):
        # 避免除零
        if target_stds[c] < 0.001:
            ratio = 1.0
        else:
            ratio = source_stds[c] / target_stds[c]
        
        # 颜色传递公式
        result_lab[:,:,c] = \
            (target_lab[:,:,c] - target_means[c]) * ratio + source_means[c]
    
    # Step 5: 将结果从 Lab 转回 RGB (逐像素)
    print("  转换回 RGB 空间...")
    result_rgb = np.zeros((h, w, 3), dtype='uint8')
    
    for y in range(h):
        for x in range(w):
            L = result_lab[y, x, 0] * 100 / 255
            A = result_lab[y, x, 1] * 256 / 255 - 128
            B = result_lab[y, x, 2] * 256 / 255 - 128
            r, g, b = lab_to_rgb_manual(L, A, B)
            result_rgb[y, x] = [b, g, r]  # BGR 格式
    
    return result_rgb

In [ ]:
# ==================== 实践2: 颜色传递 ====================

print("读取源图像 (暖色调)...")
source_img = cv2.imread("transfer_source.jpg", cv2.IMREAD_COLOR)
print("读取目标图像 (冷色调)...")
target_img = cv2.imread("transfer_target.jpg", cv2.IMREAD_COLOR)

if source_img is not None and target_img is not None:
    print(f"\n开始颜色传递...")
    print(f"源图尺寸: {source_img.shape[1]}x{source_img.shape[0]}")
    print(f"目标图尺寸: {target_img.shape[1]}x{target_img.shape[0]}")
    
    # 执行颜色传递
    # 将源图(暖色)的风格传递给目标图(冷色)
    transferred = color_transfer_manual(source_img, target_img)
    
    # 保存结果
    cv2.imwrite("color_transferred.jpg", transferred)
    print(f"\n颜色传递完成! 结果保存为: color_transferred.jpg")
else:
    print("读取图像失败!")

In [ ]:
# 颜色传递结果对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(cv2.cvtColor(source_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Source (Warm)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('Target (Cool)')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(transferred, cv2.COLOR_BGR2RGB))
axes[2].set_title('Transferred Result')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n颜色传递效果说明:")
print("  - Source (暖色调): 颜色偏暖 (R通道值高)")
print("  - Target (冷色调): 颜色偏冷 (B通道值高)")
print("  - Transferred: Target 获得了 Source 的暖色调风格")

## 四、本章总结

### 核心知识点
1. **RGB 颜色空间**：基于三原色加法混合
2. **HSV 颜色空间**：符合人眼感知，适用于颜色分割
3. **Lab 颜色空间**：感知均匀，适用于颜色比较和传递
4. **颜色传递算法**：基于 Lab 空间的统计匹配

### 关键公式
```
RGB → HSV: 线性/非线性转换 (6种区间)
RGB → Lab: sRGB→线性→XYZ→Lab (多级转换)
颜色传递: output = (input - μ_t) × (σ_s/σ_t) + μ_s
```

### 注意事项
1. OpenCV 使用 BGR 顺序
2. 颜色空间转换是非线性的，需要仔细处理
3. 颜色传递在 Lab 空间效果最好
4. 避免除零问题（标准差很小时特殊处理）
